# Benchmarking LLMs for Agentic AI
### **Building a "Team of Experts"**

In Agentic AI, we don't just use one model for everything. We build a team of agents. To build a good team, we need to know who is the fastest and who is the smartest.

**The Concept:**

We treat LLMs like "Digital Workers." To understand their value, we measure:

- **Latency**: The wait time for a response (Lower is better).

- **TPS (Tokens Per Second)**: The speed of speech/generation (Higher is better).

- **Quality**: A human-assigned score based on accuracy (1–10).

**Setting Up the Models and Modules**

In [ ]:
import os
import time
import pandas as pd
from huggingface_hub import InferenceClient
from dotenv import load_dotenv

# 1. Load your API credentials
load_dotenv()
client = InferenceClient(token=os.getenv("HF_TOKEN"), timeout=120)

# 2. Define the 'Big Three' High-Availability Models (2026 Stable Fleet)
models = {
    "qwen-2.5-7b": "Qwen/Qwen2.5-7B-Instruct",
    "llama-3.1-8b": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "gemma-2-9b": "google/gemma-2-9b-it" 
}

**The Inference Function**

This function is our "Stopwatch." It sends a message to the model and measures exactly how long it takes to finish. We calculate **TPS (Tokens Per Second)** to see which model is the most efficient worker.

In [ ]:
def run_inference(prompt, model_key):
    model_id = models[model_key]
    start_time = time.time()
    
    try:
        response = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150,
            temperature=0.7
        )
        
        latency = time.time() - start_time
        output = response.choices[0].message.content
        
        # Calculate speed (Words/Tokens per second)
        tps = len(output.split()) / latency 
        
        return {
            "model": model_key,
            "latency": round(latency, 2),
            "tps": round(tps, 2),
            "response": output,
            "status": "success"
        }
    except Exception as e:
        return {"model": model_key, "status": "failed", "error": str(e)[:50]}

In [1]:
# The Question Bank: Defining our evaluation criteria
tasks = [
    {
        "category": "Reasoning", 
        "prompt": "If a plane crashes on the border of India and Pakistan, where do you bury the survivors?"
    },
    {
        "category": "Coding", 
        "prompt": "Write a modular Python class for a database connection using the singleton pattern."
    },
    {
        "category": "Agentic Theory", 
        "prompt": "Explain the difference between a Zero-Shot Agent and a ReAct Agent in one paragraph."
    }
]

# Displaying the tasks at the beginning for transparency
print("--- Evaluation Tasks Loaded ---")
for i, task in enumerate(tasks, 1):
    print(f"Task {i} [{task['category']}]: {task['prompt']}")
print("-" * 30)

--- Evaluation Tasks Loaded ---
Task 1 [Reasoning]: If a plane crashes on the border of India and Pakistan, where do you bury the survivors?
Task 2 [Coding]: Write a modular Python class for a database connection using the singleton pattern.
Task 3 [Agentic Theory]: Explain the difference between a Zero-Shot Agent and a ReAct Agent in one paragraph.
------------------------------


**Running the Micro-Eval**

We provide specific tasks (Reasoning, Coding, etc.) and give the models a grade. This is **Human-in-the-Loop (HITL)** testing. Even if a model is fast, if the answer is wrong, we give it a low quality score.

In [2]:
eval_results = []

for task in tasks:
    print(f"\nEvaluating Category: {task['category']}")
    for m_key in models.keys():
        result = run_inference(task['prompt'], m_key)
        
        if result["status"] == "success":
            print(f"[{m_key}] Response: {result['response'][:100]}...")
            
            # Beginner interaction: Input your grade
            score = input(f"Rate {m_key} quality (1-10): ")
            result["quality_score"] = int(score) if score.isdigit() else 7
            eval_results.append(result)
        else:
            print(f"Error with {m_key}")


Evaluating Category: Reasoning


NameError: name 'models' is not defined

**Final Performance Comparison**

We use `pandas` to aggregate all our data. This tells us which model is our **Quality Champion** and which is our **Speed King**.

In [ ]:
# Convert results into a clear table
df = pd.DataFrame(eval_results)

if not df.empty:
    # Average the metrics across all tasks
    summary = df.groupby("model").agg({
        "latency": "mean",
        "tps": "mean",
        "quality_score": "mean"
    }).round(2)
    
    print("\nFINAL BENCHMARK SUMMARY")
    print(summary)
    
    # Simple logic to find the winners
    best_model = summary['quality_score'].idxmax()
    fastest_model = summary['tps'].idxmax()
    
    print(f"\nFinal Verdict:")
    print(f"- Hire {best_model} for smart tasks.")
    print(f"- Hire {fastest_model} for fast tasks.")

### Summary
1. **Pandas Aggregation**: The groupby and mean functions are used to flatten multiple task results into a single "report card" for each model.

2. **Tokens Per Second (TPS)**: This is the industry-standard way to measure the "fluidity" of an AI model.

3. **Error Handling**: The try-except block ensures that if one model (like Mistral) fails, the entire experiment doesn't crash.